In [26]:
import dspy
from dspy.evaluate import SemanticF1

from datasets import load_dataset

In [ ]:
metric = SemanticF1()
model_2 = "smollm2:360m-instruct-q4_K_M"
# model_2 = 'llama3.2:1b-instruct-q4_K_M'
lm = dspy.LM(
    # model="groq/meta-llama/llama-4-scout-17b-16e-instruct",
    # model=f"ollama_chat/{model_2}",
    temperature=0.5,
    # max_tokens=500,
)
dspy.configure(lm=lm)


In [120]:
lm.model

'groq/meta-llama/llama-4-scout-17b-16e-instruct'

In [ ]:
# model = dspy.ChainOfThought("problem -> answer")
# res = model(problem="What is 2+2")

# res

Prediction(
    reasoning='The problem asks for the sum of 2 and 2. Addition of these two identical whole numbers yields 4.',
    answer='4'
)

In [121]:
def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")["train"]
    train_split = [
        dspy.Example(
            {
                "problem": x["problem"],
                "solution": x["solution"],
                "answer": x["answer"],
            }
        ).with_inputs("problem")
        for x in train_split
    ]
    import random

    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")["train"]
    test_split = [
        dspy.Example(
            {
                "problem": x["problem"],
                "answer": x["answer"],
            }
        ).with_inputs("problem")
        for x in test_split
    ]
    test_split = test_split[:len(test_split)//4]

    train_set = train_split[: int(0.09 * tot_num)]
    val_set = train_split[int(0.09 * tot_num) :int(0.09 * tot_num)*2]
    test_set = test_split 

    return train_set, val_set, test_set

In [122]:
train_set, val_set, test_set = init_dataset()

len(train_set), len(val_set), len(test_set)

(8, 8, 7)

In [92]:
test_set[1]

Example({'problem': 'On $\\triangle ABC$ points $A, D, E$, and $B$ lie in that order on side $\\overline{AB}$ with $AD = 4$, $DE = 16$, $EB = 8$. Points $A, F, G$ and $C$ lie in that order on side $\\overline{AC}$ with $AF = 13$, $FG = 52$, and $GC = 26$. Let $M$ be the reflection of $D$ through $F$, and let $N$ be the reflection of $G$ through $E$. Quadrilateral $DEGF$ has area $288$. Find the area of heptagon $AFNBCEM$.\n\n\\begin{tikzpicture}[scale=0.07, line join=round, line cap=round, >=stealth]\n\n    \\coordinate (A) at (100,100);\n\n    \\coordinate (D) at (95,80);\n    \\coordinate (F) at (130,80);\n    \\coordinate (M) at (165,80);\n\n    \\coordinate (N) at (0,50);\n    \\coordinate (E) at (87.5,50);\n    \\coordinate (G) at (175,50);\n\n    \\coordinate (B) at ($(D)!2!(E)$);\n    \\coordinate (C) at ($(F)!2!(G)$);\n    \n    \\fill[draw=black, fill=gray!20] (N) -- (E) -- (M) -- (F) -- cycle;\n    \\fill[draw=black, fill=gray!20] (N) -- (E) -- (C) -- (B) -- cycle;\n    \\fil

In [76]:
print("Problem:")
print(train_set[0]['problem'])
print("\n\nSolution:")
print(train_set[0]['solution'])
print("\n\nAnswer:")
print(train_set[0]['answer'])

Problem:
In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.


Solution:
We have the following diagram:

Let $X$ and $W$ be the points where $AP$ and $BQ$ extend to meet $CD$, and $YZ$ be the height of $\triangle AZB$. As proven in Solution 2, triangles $APD$ and $DPW$ are congruent right triangles. Therefore, $AD = DW = 333$. We can apply this logic to triangles $BCQ$ and $XCQ$ as well, giving us $BC = CX = 333$. Since $CD = 650$, $XW = DW + CX - CD = 16$.
Additionally, we can see that $\triangle XZW$ is similar to $\triangle PQZ$ and $\triangle AZB$. We know that $\frac{XW}{AB} = \frac{16}{500}$. So, we can say that the height of the triangle $AZB$ is $500u$ while the height of the triangle $XZW$ is $16u$. After that, we can figure out the distance from 

In [123]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

In [94]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError as e:
        return 0
    return int(correct_answer == llm_answer)

In [124]:
# len(test_set[])
# test_set[8]

program(problem = test_set[0]['problem'])


Prediction(
    reasoning='To solve this problem, we first need to understand what $17_b$ and $97_b$ mean. These represent numbers in base $b$, where $17_b = 1 cdot b + 7$ and $97_b = 9 cdot b + 7$. We are looking for integer bases $b > 9$ where $17_b$ is a divisor of $97_b$. This means $97_b$ must be divisible by $17_b$ without a remainder. So, we need to find all $b > 9$ where $\x0crac{9b + 7}{b + 7}$ is an integer. We can simplify this expression: $\x0crac{9b + 7}{b + 7} = \x0crac{9(b + 7) - 56}{b + 7} = 9 - \x0crac{56}{b + 7}$. For this to be an integer, $b + 7$ must be a divisor of 56. The divisors of 56 are 1, 2, 4, 7, 8, 14, 28, and 56. Since $b > 9$, $b + 7 > 16$. Therefore, the possible values for $b + 7$ are 28 and 56, leading to $b = 21$ or $b = 49$. We need to verify if both are valid and then sum them.',
    answer='21 + 49 = 70'
)

In [96]:
test_set[0]

Example({'problem': 'Find the sum of all integer bases $b>9$ for which $17_b$ is a divisor of $97_b.$', 'answer': 70}) (input_keys={'problem'})

In [125]:
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=1,
    display_table=True,
    display_progress=True
)

evaluate(program)

  0%|          | 0/7 [00:00<?, ?it/s]

2026/02/24 17:32:18 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'Find the sum of all integer bases $b>9$ for which $17_b$ is a divisor of $97_b.$', 'answer': 70}) (input_keys={'problem'}): 'Example' object has no attribute 'question'. Set `provide_traceback=True` for traceback.


Average Metric: 0.00 / 0 (0%):   0%|          | 0/7 [00:00<?, ?it/s]

2026/02/24 17:32:19 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'On $\\triangle ABC$ points $A, D, E$, and $B$ lie in that order on side $\\overline{AB}$ with $AD = 4$, $DE = 16$, $EB = 8$. Points $A, F, G$ and $C$ lie in that order on side $\\overline{AC}$ with $AF = 13$, $FG = 52$, and $GC = 26$. Let $M$ be the reflection of $D$ through $F$, and let $N$ be the reflection of $G$ through $E$. Quadrilateral $DEGF$ has area $288$. Find the area of heptagon $AFNBCEM$.\n\n\\begin{tikzpicture}[scale=0.07, line join=round, line cap=round, >=stealth]\n\n    \\coordinate (A) at (100,100);\n\n    \\coordinate (D) at (95,80);\n    \\coordinate (F) at (130,80);\n    \\coordinate (M) at (165,80);\n\n    \\coordinate (N) at (0,50);\n    \\coordinate (E) at (87.5,50);\n    \\coordinate (G) at (175,50);\n\n    \\coordinate (B) at ($(D)!2!(E)$);\n    \\coordinate (C) at ($(F)!2!(G)$);\n    \n    \\fill[draw=black, fill=gray!20] (N) -- (E) -- (M) -- (F) -- cycle;\n    \\fill[draw=black

Average Metric: 0.00 / 0 (0%):  29%|██▊       | 2/7 [00:01<00:03,  1.31it/s]

...

2026/02/24 17:32:33 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'The twelve letters $A$,$B$,$C$,$D$,$E$,$F$,$G$,$H$,$I$,$J$,$K$, and $L$ are randomly grouped into six pairs of letters. The two letters in each pair are placed next to each other in alphabetical order to form six two-letter words, and then those six words are listed alphabetically. For example, a possible result is $AB$, $CJ$, $DG$, $EK$, $FL$, $HI$. The probability that the last word listed contains $G$ is $\\frac mn$, where $m$ and $n$ are relatively prime positive integers. Find $m+n$.', 'answer': 821}) (input_keys={'problem'}): 'Example' object has no attribute 'question'. Set `provide_traceback=True` for traceback.


Average Metric: 0.00 / 0 (0%): 100%|██████████| 7/7 [00:14<00:00,  2.08s/it]

2026/02/24 17:32:33 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 7 (0.0%)


,problem,answer,SemanticF1
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,✔️ [0.000]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,✔️ [0.000]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,✔️ [0.000]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,✔️ [0.000]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,✔️ [0.000]
5,An isosceles trapezoid has an inscribed circle tangent to each of ...,504,✔️ [0.000]
6,"The twelve letters $A$,$B$,$C$,$D$,$E$,$F$,$G$,$H$,$I$,$J$,$K$, an...",821,✔️ [0.000]


EvaluationResult(score=0.0, results=<list of 7 results>)

In [126]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    try:
        llm_answer = int(prediction.answer)
    except ValueError as e:
        feedback_text = f"The final answer must be a valid integer and nothing else. You responded with '{prediction.answer}', which couldn't be parsed as a python integer. Please ensure your answer is a valid integer without any additional text or formatting."
        feedback_text += f" The correct answer is '{correct_answer}'."
        if written_solution:
            feedback_text += f" Here's the full step-by-step solution:\n{written_solution}\n\nThink about what takeaways you can learn from this solution to improve your future answers and approach to similar problems and ensure your final answer is a valid integer."
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)

    feedback_text = ""
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{correct_answer}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{correct_answer}'."
    
    if written_solution:
        feedback_text += f" Here's the full step-by-step solution:\n{written_solution}\n\nThink about what takeaways you can learn from this solution to improve your future answers and approach to similar problems."

    return dspy.Prediction(score=score, feedback=feedback_text)

In [127]:
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=1,
    track_stats=True,
    reflection_minibatch_size=2,
    reflection_lm=dspy.LM(
        model="groq/llama-3.1-8b-instant",
        temperature=0.7,
        # max_tokens=320,
    ),
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2026/02/24 17:33:04 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 412 metric calls of the program. This amounts to 25.75 full evals on the train+val set.
2026/02/24 17:33:04 INFO dspy.teleprompt.gepa.gepa: Using 8 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/412 [00:00<?, ?rollouts/s]2026/02/24 17:33:28 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 8 (37.5%)
2026/02/24 17:33:28 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.375 over 8 / 8 examples
GEPA Optimization:   2%|▏         | 8/412 [00:24<20:25,  3.03s/rollouts]2026/02/24 17:33:28 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.375


Average Metric: 0.00 / 2 (0.0%): 100%|██████████| 2/2 [00:06<00:00,  3.31s/it]

2026/02/24 17:33:35 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 2 (0.0%)


2026/02/24 17:33:36 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for predict: # New Instructions

## Task Description
The task involves solving a problem and providing the answer in the correct format. Problems may involve mathematical equations, logic puzzles, or other types of reasoning. The task may require the assistant to identify niche and domain-specific factual information, use generalizable strategies to solve the problem, and provide a valid integer answer.

## Input Format
The input will be a problem statement, which may include mathematical equations, logic statements, or other types of reasoning. The input may also include constraints, such as the format of the answer or the type of solution required.

## Task Requirements
1. Carefully read the input problem statement and identify the task requirements.
2. Infer detailed task description about the task and identify niche and domain-specific factual information.
3. Use generalizable strategies to solve the

  0%|          | 0/2 [00:00<?, ?it/s]

...

2026/02/24 17:54:16 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 2 (0.0%)


2026/02/24 17:54:19 INFO dspy.teleprompt.gepa.gepa: Iteration 100: Exception during reflection/proposal: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kgks8g58ewcad2jre1k9yrms` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 2366, Requested 5036. Please try again in 14.02s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

2026/02/24 17:54:19 INFO dspy.teleprompt.gepa.gepa: Traceback (most recent call last):
  File "/home/esai/Desktop/python/dev/lib/python3.10/site-packages/litellm/llms/custom_httpx/llm_http_handler.py", line 235, in _make_common_sync_call
    response = sync_httpx_client.post(
  File "/home/esai/Desktop/python/dev/lib/python3.10/site-packages/litellm/llms/custom_httpx/http_handler.py", line 1043, in post
    raise e
  File "/home/esai/Desktop/python/dev/li

Average Metric: 0.00 / 2 (0.0%): 100%|██████████| 2/2 [00:00<00:00, 275.26it/s]

2026/02/24 17:54:19 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 2 (0.0%)


2026/02/24 17:54:24 INFO dspy.teleprompt.gepa.gepa: Iteration 101: Exception during reflection/proposal: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kgks8g58ewcad2jre1k9yrms` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 1887, Requested 4920. Please try again in 8.07s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

2026/02/24 17:54:24 INFO dspy.teleprompt.gepa.gepa: Traceback (most recent call last):
  File "/home/esai/Desktop/python/dev/lib/python3.10/site-packages/litellm/llms/custom_httpx/llm_http_handler.py", line 235, in _make_common_sync_call
    response = sync_httpx_client.post(
  File "/home/esai/Desktop/python/dev/lib/python3.10/site-packages/litellm/llms/custom_httpx/http_handler.py", line 1043, in post
    raise e
  File "/home/esai/Desktop/python/dev/lib

In [129]:
optimized_program.inspect_history()


In [113]:
optimized_program(problem = test_set[-1])

2026/02/24 17:24:40 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=None. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.5)  if the reason for truncation is repetition.


Prediction(
    reasoning="We are grouping 12 distinct letters into 6 unordered pairs. The total number of ways to do this is:\n\nTotal pairings = 12! / (2^6 * 6!) = 10395.\n\nWe want the probability that the last word (when the 6 two-letter words are listed alphabetically) contains the letter G. Since each word is written in alphabetical order within the pair, and the six words are listed alphabetically, the last word is the one whose first letter is largest among the six first letters.\n\nG is the 7th letter alphabetically. For G to appear in the last word, G must be paired with a letter that comes after G (so that the word starts with G), and moreover, no letter after G can be the first letter of any other word. That is, H, I, J, K, L must all be paired with letters that come before them (so their words start with their partners), and G must be paired with one of H, I, J, K, L.\n\nSo, the condition is:\n- G is paired with one of {H, I, J, K, L} (5 choices), forming a word starting w

In [130]:
optimized_program(problem = test_set[-1])

Prediction(
    reasoning="## Step 1: Understand the problem and what is being asked.\nWe are given twelve letters $A$,$B$,$C$,$D$,$E$,$F$,$G$,$H$,$I$,$J$,$K$, and $L$ that are randomly grouped into six pairs of letters. These pairs are then arranged in alphabetical order to form six two-letter words, which are listed alphabetically. We need to find the probability that the last word listed contains $G$ and then derive the sum of the numerator and denominator of this probability expressed as $\x0crac{m}{n}$, where $m$ and $n$ are relatively prime positive integers.\n\n## Step 2: Determine the total number of possible pairings of the letters.\nThere are $\x08inom{12}{2}$ ways to choose the first pair, $\x08inom{10}{2}$ ways to choose the second pair, and so on, until there is only one pair left. However, because the order of choosing the pairs does not matter, we need to divide by the number of ways to order the 6 pairs, which is $6!$. This results in a total of $\x0crac{\x08inom{12}{2}

In [131]:
optimized_program.inspect_history()





[2026-02-24T17:57:13.331511]

System message:

Your input fields are:
1. `problem` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## problem ## ]]
{problem}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "answer": "{answer}"
}
In adhering to this structure, your objective is: 
        Solve the problem and provide the answer in the correct format.


User message:

[[ ## problem ## ]]
Example({'problem': 'The twelve letters $A$,$B$,$C$,$D$,$E$,$F$,$G$,$H$,$I$,$J$,$K$, and $L$ are randomly grouped into six pairs of letters. The two letters in each pair are placed next to each other in alphabetical order to form six two-letter words, and then those six words are listed alphabetically. For example, a possible result is $AB$, $CJ$, $DG$, $EK$, $FL$, $HI$. The probability tha

In [ ]:
optimized_program.save('opti.json')